# ML-ансамбли tuned-моделей



In [ ]:
import os
import gc
import ast
import json
import random
import re
from datetime import datetime
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from scipy.optimize import nnls, minimize
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge,
    HuberRegressor,
    QuantileRegressor,
)
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"
DATA_PATH_ENV = "DATA_FINALL_PATH"
TUNED_SUMMARY_PATH_ENV = "ML_TUNING_SUMMARY_PATH"

RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BLEND_FIT_FRAC = 0.50
FORCE_REBUILD_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True

USE_ALL_TUNED_MODELS = True
MANUAL_MODEL_POOL = []

RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 7
REFIT_TOP_ENSEMBLES = 250
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

ART_DIR = Path("./artifacts_ml_ensembles_final")
ART_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = ART_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_CACHE_DIR = RUN_DIR / "pred_cache"
for sub in ["split_val", "split_test_typical", "split_test_full", "fullfit_test_typical", "fullfit_test_full"]:
    (PRED_CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("ART_DIR:", ART_DIR)
print("RUN_DIR:", RUN_DIR)


In [ ]:
def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)

def serialize_value(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, Path):
        return str(v)
    if pd.isna(v) and not isinstance(v, (str, bytes, bool)):
        return None
    return v

def serialize_params(obj):
    if isinstance(obj, dict):
        return {str(k): serialize_params(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [serialize_params(v) for v in obj]
    return serialize_value(obj)

def parse_best_params(raw):
    if isinstance(raw, dict):
        return serialize_params(raw)

    if isinstance(raw, (list, tuple)):
        return serialize_params(raw)

    if raw is None:
        return {}

    try:
        if pd.isna(raw):
            return {}
    except Exception:
        pass

    s = str(raw).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return {}

    # Сначала пробуем JSON.
    try:
        return serialize_params(json.loads(s))
    except Exception:
        pass

    # Затем строковое представление Python-словаря.
    try:
        return serialize_params(ast.literal_eval(s))
    except Exception:
        pass

    # Для CSV-строк отдельно нормализуем null/NaN/true/false.
    s2 = re.sub(r"\bNaN\b", "None", s, flags=re.IGNORECASE)
    s2 = re.sub(r"\bnull\b", "None", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\btrue\b", "True", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bfalse\b", "False", s2, flags=re.IGNORECASE)

    try:
        return serialize_params(ast.literal_eval(s2))
    except Exception as e:
        preview = s[:300].replace("\n", " ")
        raise ValueError(
            f"Не удалось распарсить колонку params. Preview: {preview}"
        ) from e

def sanitize_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in name)

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def calc_reg_metrics(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MdAE": mdae(y_true, y_pred),
    }

def cleanup_memory():
    gc.collect()


In [ ]:
tuned_summary_value = os.environ.get(TUNED_SUMMARY_PATH_ENV)

TUNED_SUMMARY_PATH = Path(tuned_summary_value).expanduser()

if TUNED_SUMMARY_PATH.suffix.lower() == ".json":
    with open(TUNED_SUMMARY_PATH, "r", encoding="utf-8") as f:
        tuned_df = pd.DataFrame(json.load(f))
else:
    tuned_df = pd.read_csv(TUNED_SUMMARY_PATH)

if "params" not in tuned_df.columns:
    raise ValueError("В tuned summary нет колонки `params`.")

print(f"TUNED_SUMMARY_PATH: {TUNED_SUMMARY_PATH}")
display(tuned_df[["model", "selected_from", "cv_MAE", "holdout_typical_MAE", "holdout_full_MAE"]])

data_path_value = os.environ.get(DATA_PATH_ENV)
if not data_path_value:
    raise RuntimeError(f"Перед запуском укажи путь к датасету в переменной окружения {DATA_PATH_ENV}.")

DATA_PATH = Path(data_path_value).expanduser()
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Файл из {DATA_PATH_ENV} не найден: {DATA_PATH}")

print(f"DATA_PATH: {DATA_PATH}")


## Разбор параметров моделей


In [ ]:
seed_everything(SEED)

data = pd.read_csv(DATA_PATH)
split = int(len(data) * 0.8)

train_full = data.iloc[:split].copy()
test_full = data.iloc[split:].copy()

train_typical = train_full[train_full[TARGET_COL] < DURATION_CAP].copy()
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy()

Xtrain = train_typical.drop(columns=[TARGET_COL], errors="ignore")
ytrain = train_typical[TARGET_COL].reset_index(drop=True)
Xtest_typical = test_typical.drop(columns=[TARGET_COL], errors="ignore")
ytest_typical = test_typical[TARGET_COL].reset_index(drop=True)
Xtest_full = test_full.drop(columns=[TARGET_COL], errors="ignore")
ytest_full = test_full[TARGET_COL].reset_index(drop=True)

# Внутренний temporal split для подбора ансамбля.
blend_split_idx = int(len(Xtrain) * 0.85)
X_blend_tr = Xtrain.iloc[:blend_split_idx].copy()
y_blend_tr = ytrain.iloc[:blend_split_idx].copy()
X_blend_val = Xtrain.iloc[blend_split_idx:].copy()
y_blend_val = ytrain.iloc[blend_split_idx:].copy()

print(f"Train raw: {len(train_full)}")
print(f"Train typical < {DURATION_CAP}: {len(train_typical)}")
print(f"Blend train: {len(X_blend_tr)}")
print(f"Blend validation: {len(X_blend_val)}")
print(f"Holdout typical: {len(test_typical)}")
print(f"Holdout full: {len(test_full)}")
print(f"Holdout outliers >= {DURATION_CAP}: {(test_full[TARGET_COL] >= DURATION_CAP).sum()}")

if len(X_blend_val) == 0:
    raise ValueError("Слишком маленький train_typical для внутреннего split ансамбля.")


## Восстановление tuned ML-моделей

В пул ансамблирования попадают только модели из финальной сводки тюнинга:
- `BayRidge`
- `Lasso`
- `Elastic`
- `Ridge`
- `Hub-Reg`
- `Gboost-Reg`
- `XGB_reg`

Baseline-модели без тюнинга в этот пул не добавляются.


In [ ]:
def make_ml_estimator(model_name: str, params: dict):
    p = serialize_params(params)

    if model_name == "BayRidge":
        model = BayesianRidge(
            alpha_1=float(p.get("alpha_1", 1e-6)),
            alpha_2=float(p.get("alpha_2", 1e-6)),
            lambda_1=float(p.get("lambda_1", 1e-6)),
            lambda_2=float(p.get("lambda_2", 1e-6)),
            max_iter=int(p.get("max_iter", 300)),
            tol=float(p.get("tol", 1e-3)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            compute_score=bool(p.get("compute_score", False)),
            verbose=bool(p.get("verbose", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Lasso":
        model = Lasso(
            alpha=float(p.get("alpha", 1.0)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 100000)),
            tol=float(p.get("tol", 1e-4)),
            random_state=p.get("random_state", SEED),
            selection=p.get("selection", "cyclic"),
            warm_start=bool(p.get("warm_start", False)),
            positive=bool(p.get("positive", False)),
            precompute=bool(p.get("precompute", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Elastic":
        model = ElasticNet(
            alpha=float(p.get("alpha", 1.0)),
            l1_ratio=float(p.get("l1_ratio", 0.5)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 100000)),
            tol=float(p.get("tol", 1e-4)),
            random_state=p.get("random_state", SEED),
            selection=p.get("selection", "cyclic"),
            warm_start=bool(p.get("warm_start", False)),
            positive=bool(p.get("positive", False)),
            precompute=bool(p.get("precompute", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Ridge":
        model = Ridge(
            alpha=float(p.get("alpha", 1.0)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            copy_X=bool(p.get("copy_X", True)),
            max_iter=None if p.get("max_iter") is None else int(p.get("max_iter")),
            tol=float(p.get("tol", 1e-4)),
            solver=p.get("solver", "auto"),
            positive=bool(p.get("positive", False)),
            random_state=p.get("random_state", None),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Hub-Reg":
        model = HuberRegressor(
            epsilon=float(p.get("epsilon", 1.35)),
            alpha=float(p.get("alpha", 1e-4)),
            fit_intercept=bool(p.get("fit_intercept", True)),
            max_iter=int(p.get("max_iter", 1000)),
            tol=float(p.get("tol", 1e-5)),
            warm_start=bool(p.get("warm_start", False)),
        )
        return Pipeline([("scaler", StandardScaler()), ("model", model)])

    if model_name == "Gboost-Reg":
        model = GradientBoostingRegressor(
            loss=p.get("loss", "absolute_error"),
            learning_rate=float(p.get("learning_rate", 0.1)),
            n_estimators=int(p.get("n_estimators", 100)),
            subsample=float(p.get("subsample", 1.0)),
            criterion=p.get("criterion", "friedman_mse"),
            min_samples_split=int(p.get("min_samples_split", 2)),
            min_samples_leaf=int(p.get("min_samples_leaf", 1)),
            min_weight_fraction_leaf=float(p.get("min_weight_fraction_leaf", 0.0)),
            max_depth=int(p.get("max_depth", 3)),
            min_impurity_decrease=float(p.get("min_impurity_decrease", 0.0)),
            init=p.get("init", None),
            random_state=p.get("random_state", SEED),
            max_features=p.get("max_features", None),
            alpha=float(p.get("alpha", 0.9)),
            verbose=int(p.get("verbose", 0)),
            max_leaf_nodes=p.get("max_leaf_nodes", None),
            warm_start=bool(p.get("warm_start", False)),
            validation_fraction=float(p.get("validation_fraction", 0.1)),
            n_iter_no_change=p.get("n_iter_no_change", None),
            tol=float(p.get("tol", 1e-4)),
            ccp_alpha=float(p.get("ccp_alpha", 0.0)),
        )
        return model

    if model_name == "XGB_reg":
        if not HAS_XGB:
            raise RuntimeError("XGBoost недоступен в окружении.")
        xgb_params = {
            "objective": p.get("objective", "reg:absoluteerror"),
            "eval_metric": p.get("eval_metric", "mae"),
            "n_estimators": int(p.get("n_estimators", 500)),
            "learning_rate": float(p.get("learning_rate", 0.05)),
            "max_depth": int(p.get("max_depth", 3)),
            "min_child_weight": float(p.get("min_child_weight", 1.0)),
            "subsample": float(p.get("subsample", 1.0)),
            "colsample_bytree": float(p.get("colsample_bytree", 1.0)),
            "gamma": float(p.get("gamma", 0.0)),
            "reg_alpha": float(p.get("reg_alpha", 0.0)),
            "reg_lambda": float(p.get("reg_lambda", 1.0)),
            "max_bin": int(p.get("max_bin", 256)),
            "tree_method": p.get("tree_method", "hist"),
            "n_jobs": int(p.get("n_jobs", -1)),
            "random_state": p.get("random_state", SEED),
        }
        return XGBRegressor(**xgb_params)

    raise KeyError(model_name)

tuned_df["best_params_parsed"] = tuned_df["params"].apply(parse_best_params)

if USE_ALL_TUNED_MODELS:
    pool_df = tuned_df.copy().reset_index(drop=True)
elif MANUAL_MODEL_POOL:
    pool_df = tuned_df[tuned_df["model"].isin(MANUAL_MODEL_POOL)].copy().reset_index(drop=True)
else:
    pool_df = tuned_df.copy().reset_index(drop=True)

SELECTED_MODELS = pool_df["model"].tolist()
print(f"Выбрано tuned ML-моделей: {len(SELECTED_MODELS)}")
print(SELECTED_MODELS)
display(pool_df[["model", "selected_from", "cv_MAE", "holdout_typical_MAE", "holdout_full_MAE"]])


In [ ]:
def _pred_path(kind: str, model_name: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(model_name)}.npy"

def _save_pred(kind: str, model_name: str, arr):
    np.save(_pred_path(kind, model_name), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, model_name: str):
    path = _pred_path(kind, model_name)
    if not path.exists():
        return None
    return np.load(path)

def _have_all_cached(model_name: str) -> bool:
    needed = [
        _pred_path("split_val", model_name),
        _pred_path("split_test_typical", model_name),
        _pred_path("split_test_full", model_name),
        _pred_path("fullfit_test_typical", model_name),
        _pred_path("fullfit_test_full", model_name),
    ]
    return all(p.exists() for p in needed)

def fit_and_predict_tuned_model(row: pd.Series):
    model_name = row["model"]
    best_params = row["best_params_parsed"]

    print(f"\n=== Восстановление {model_name} ===")
    print(best_params)
    t0 = datetime.now()

    if RESUME_IF_PREDICTIONS_EXIST and not FORCE_REBUILD_BASE_PREDICTIONS and _have_all_cached(model_name):
        print("Используются сохранённые предсказания.")
        return {
            "model": model_name,
            "status": "cached",
            "elapsed_sec": 0.0,
        }

    seed_everything(SEED)

    # Обучение на части train для внутренней validation ансамбля.
    model_split = make_ml_estimator(model_name, best_params)
    model_split.fit(X_blend_tr, y_blend_tr)

    pred_val = clip_pred(model_split.predict(X_blend_val))
    pred_test_typ_split = clip_pred(model_split.predict(Xtest_typical))
    pred_test_full_split = clip_pred(model_split.predict(Xtest_full))

    # Обучение на всём train для финальной оценки holdout.
    model_full = make_ml_estimator(model_name, best_params)
    model_full.fit(Xtrain, ytrain)

    pred_test_typ_fullfit = clip_pred(model_full.predict(Xtest_typical))
    pred_test_full_fullfit = clip_pred(model_full.predict(Xtest_full))

    _save_pred("split_val", model_name, pred_val)
    _save_pred("split_test_typical", model_name, pred_test_typ_split)
    _save_pred("split_test_full", model_name, pred_test_full_split)
    _save_pred("fullfit_test_typical", model_name, pred_test_typ_fullfit)
    _save_pred("fullfit_test_full", model_name, pred_test_full_fullfit)

    elapsed = (datetime.now() - t0).total_seconds()
    return {
        "model": model_name,
        "status": "rebuilt",
        "elapsed_sec": float(elapsed),
        "split_val_MAE": mae(y_blend_val, pred_val),
        "fullfit_typical_MAE": mae(ytest_typical, pred_test_typ_fullfit),
        "fullfit_full_MAE": mae(ytest_full, pred_test_full_fullfit),
    }

def build_prediction_matrices(pool_frame: pd.DataFrame):
    rows = []
    for _, row in pool_frame.iterrows():
        try:
            out = fit_and_predict_tuned_model(row)
            rows.append(out)
        except Exception as e:
            rows.append({
                "model": row["model"],
                "status": "failed",
                "error": repr(e),
            })
            print(f"Ошибка восстановления модели {row['model']}: {e}")
        cleanup_memory()
    return pd.DataFrame(rows)

def load_prediction_frame(kind: str, model_names):
    cols = {}
    for m in model_names:
        arr = _load_pred(kind, m)
        if arr is not None:
            cols[m] = arr
    return pd.DataFrame(cols)

def get_base_leaderboard_from_predictions(pred_df: pd.DataFrame, y_true=None):
    y_true = y_blend_val if y_true is None else np.asarray(y_true, dtype=float)
    rows = []
    for col in pred_df.columns:
        rows.append({
            "model": col,
            "MAE": mae(y_true, pred_df[col].values),
            "RMSE": rmse(y_true, pred_df[col].values),
            "MdAE": mdae(y_true, pred_df[col].values),
        })
    return pd.DataFrame(rows).sort_values(["MAE", "RMSE", "MdAE"], kind="stable").reset_index(drop=True)


In [ ]:
def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
        def objective(w):
            return mae(y_fit, X @ w)
        res = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=[cons], options={"maxiter": 500, "ftol": 1e-8})
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return Ridge(alpha=1.0)
    elif stacker_name == "lasso":
        return Lasso(alpha=1e-3, max_iter=20000, random_state=SEED)
    elif stacker_name == "elastic":
        return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=20000, random_state=SEED)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=2000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(loss="absolute_error", random_state=SEED, n_estimators=300, learning_rate=0.05, max_depth=2, min_samples_leaf=5, min_samples_split=4, subsample=0.9)
    elif stacker_name == "rf":
        return RandomForestRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "etr":
        return ExtraTreesRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "xgb":
        if not HAS_XGB:
            raise RuntimeError("xgboost недоступен")
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=2,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    return clip_pred(fitted["model"].predict(pred_df.values.astype(float)))

def fit_pair_weight_grid(pred_fit: pd.DataFrame, y_fit):
    if pred_fit.shape[1] != 2:
        raise ValueError("pair_grid expects exactly 2 models")
    best = None
    a = pred_fit.iloc[:, 0].values.astype(float)
    b = pred_fit.iloc[:, 1].values.astype(float)
    for w in PAIR_WEIGHT_GRID:
        pred = clip_pred(w * a + (1.0 - w) * b)
        score = mae(y_fit, pred)
        if best is None or score < best["selection_MAE"]:
            best = {"kind": "pair_grid", "weight_first": float(w), "selection_MAE": score}
    return best

def predict_pair_weight_grid(fitted, pred_df: pd.DataFrame):
    w = float(fitted["weight_first"])
    a = pred_df.iloc[:, 0].values.astype(float)
    b = pred_df.iloc[:, 1].values.astype(float)
    return clip_pred(w * a + (1.0 - w) * b)

def score_predictions(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MdAE": mdae(y_true, y_pred),
    }


In [ ]:
def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    fit_rank_df = get_base_leaderboard_from_predictions(pred_fit_df, y_fit)
    ranked_models = fit_rank_df["model"].tolist()
    return ranked_models, fit_rank_df

def spec_tag(spec):
    models_part = "__".join(spec["models"])
    return f'{spec["family"]}::{spec["name"]}::{models_part}'

def evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fit_sub = pred_fit_df[models].copy()
    sel_sub = pred_sel_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_sel = clip_pred(sel_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_sel = aggregate_predictions(sel_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(fit_sub, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, sel_sub)
        out = {
            "family": family, "name": name, "models": models, "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(fit_sub, y_fit)
        pred_sel = predict_pair_weight_grid(fitted, sel_sub)
        out = {
            "family": family, "name": name, "models": models, "n_models": len(models),
            "weight_first": fitted["weight_first"],
        }
    elif family == "stacker":
        fitted = fit_stacker(fit_sub, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, sel_sub)
        out = {"family": family, "name": name, "models": models, "n_models": len(models), "stacker_name": spec["stacker_name"]}
    else:
        raise KeyError(family)

    sel_metrics = score_predictions(y_sel, pred_sel)
    out["selection_MAE"] = sel_metrics["MAE"]
    out["selection_RMSE"] = sel_metrics["RMSE"]
    out["selection_MdAE"] = sel_metrics["MdAE"]
    out["tag"] = spec_tag(spec)
    out["spec"] = spec
    return out

def refit_and_eval_spec(spec, pred_val_df, y_val, pred_test_typ_df, y_test_typ, pred_test_full_df, y_test_full):
    models = spec["models"]
    val_sub = pred_val_df[models].copy()
    test_typ_sub = pred_test_typ_df[models].copy()
    test_full_sub = pred_test_full_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_typ = clip_pred(test_typ_sub.iloc[:, 0].values)
        pred_full = clip_pred(test_full_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_typ = aggregate_predictions(test_typ_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(test_full_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(val_sub, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, test_typ_sub)
        pred_full = predict_weighted_blender(fitted, test_full_sub)
        out = {
            "family": family, "name": name, "models": models, "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(val_sub, y_val)
        pred_typ = predict_pair_weight_grid(fitted, test_typ_sub)
        pred_full = predict_pair_weight_grid(fitted, test_full_sub)
        out = {
            "family": family, "name": name, "models": models, "n_models": len(models),
            "weight_first": fitted["weight_first"],
            "weights_like": [fitted["weight_first"], 1.0 - fitted["weight_first"]],
        }
    elif family == "stacker":
        fitted = fit_stacker(val_sub, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, test_typ_sub)
        pred_full = predict_stacker(fitted, test_full_sub)
        weights_like = None
        if hasattr(fitted["model"], "coef_"):
            coef = np.ravel(getattr(fitted["model"], "coef_"))
            if len(coef) == len(models):
                weights_like = coef.tolist()
        out = {
            "family": family, "name": name, "models": models, "n_models": len(models),
            "weights_like": weights_like or [],
        }
    else:
        raise KeyError(family)

    typ_metrics = score_predictions(y_test_typ, pred_typ)
    full_metrics = score_predictions(y_test_full, pred_full)

    out.update({
        "tag": spec_tag(spec),
        "MAE_typical": typ_metrics["MAE"],
        "RMSE_typical": typ_metrics["RMSE"],
        "MdAE_typical": typ_metrics["MdAE"],
        "MAE_full": full_metrics["MAE"],
        "RMSE_full": full_metrics["RMSE"],
        "MdAE_full": full_metrics["MdAE"],
    })
    return out

def greedy_forward_specs(ranked_models):
    specs = []
    current = []
    remaining = ranked_models.copy()
    while remaining:
        best_candidate = None
        for m in remaining:
            cand = current + [m]
            specs.append({"family": "aggregate", "name": f"greedy_mean_step{len(cand)}", "models": cand, "agg_method": "mean"})
            specs.append({"family": "weighted", "name": f"greedy_inverse_step{len(cand)}", "models": cand, "weight_scheme": "inverse_mae"})
            best_candidate = m
        current = current + [best_candidate]
        remaining = [m for m in remaining if m != best_candidate]
    return specs


In [ ]:
# 1. Восстановление tuned-моделей и сохранение предсказаний
rebuild_log_df = build_prediction_matrices(pool_df)
display(rebuild_log_df)
rebuild_log_df.to_csv(RUN_DIR / "rebuild_log.csv", index=False)
rebuild_log_df.to_csv(ART_DIR / "rebuild_log_latest.csv", index=False)

# Матрицы предсказаний.
val_pred_df = load_prediction_frame("split_val", SELECTED_MODELS)
test_typ_split_pred_df = load_prediction_frame("split_test_typical", SELECTED_MODELS)
test_full_split_pred_df = load_prediction_frame("split_test_full", SELECTED_MODELS)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", SELECTED_MODELS)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", SELECTED_MODELS)

available_models = sorted(
    set(val_pred_df.columns)
    & set(test_typ_fullfit_pred_df.columns)
    & set(test_full_fullfit_pred_df.columns),
    key=lambda m: SELECTED_MODELS.index(m),
)

val_pred_df = val_pred_df[available_models].copy()
test_typ_split_pred_df = test_typ_split_pred_df[available_models].copy()
test_full_split_pred_df = test_full_split_pred_df[available_models].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_models].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_models].copy()

print(f"Доступно восстановленных моделей: {len(available_models)}")
print(available_models)

if len(available_models) < 2:
    raise RuntimeError("Для ансамблей нужно хотя бы 2 успешно восстановленные модели.")

val_pred_df.to_csv(RUN_DIR / "base_pred_split_val.csv", index=False)
val_pred_df.to_csv(ART_DIR / "base_pred_split_val_latest.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_typ_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_typical_latest.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)
test_full_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_full_latest.csv", index=False)

y_val = np.asarray(y_blend_val, dtype=float)
y_test_typ = np.asarray(ytest_typical, dtype=float)
y_test_full = np.asarray(ytest_full, dtype=float)

base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df)
display(base_leaderboard.head(15))
base_leaderboard.to_csv(RUN_DIR / "rebuilt_base_leaderboard.csv", index=False)
base_leaderboard.to_csv(ART_DIR / "rebuilt_base_leaderboard_latest.csv", index=False)

blend_cut = int(len(y_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")
pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_val[:blend_cut].copy()
y_sel = y_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
print("Лучшие модели на blend-fit по MAE:")
display(fit_rank_df.head(15))
fit_rank_df.to_csv(RUN_DIR / "blend_fit_model_ranking.csv", index=False)
fit_rank_df.to_csv(ART_DIR / "blend_fit_model_ranking_latest.csv", index=False)

# 2. Формирование набора ансамблей
specs = []

# Одиночные модели.
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# Агрегации по всему пулу моделей.
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
    {"family": "weighted", "name": "all_inverse_mae", "models": all_models, "weight_scheme": "inverse_mae"},
    {"family": "weighted", "name": "all_inverse_mae_sq", "models": all_models, "weight_scheme": "inverse_mae_sq"},
    {"family": "weighted", "name": "all_softmax_t1", "models": all_models, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}},
    {"family": "weighted", "name": "all_nnls", "models": all_models, "weight_scheme": "nnls"},
    {"family": "weighted", "name": "all_simplex_mae", "models": all_models, "weight_scheme": "simplex_mae"},
])

# Ансамбли из top-k моделей.
if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankpow_{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex_mae", "models": topk, "weight_scheme": "simplex_mae"})

# Все пары моделей.
if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        specs.append({"family": "weighted", "name": "pair_nnls", "models": [a, b], "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": "pair_simplex", "models": [a, b], "weight_scheme": "simplex_mae"})

# Все тройки моделей.
if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        trio = list(trio)
        specs.append({"family": "aggregate", "name": "triple_mean", "models": trio, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": "triple_median", "models": trio, "agg_method": "median"})
        specs.append({"family": "weighted", "name": "triple_inverse", "models": trio, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": "triple_nnls", "models": trio, "weight_scheme": "nnls"})

# Перебор подмножеств среди top-N моделей.
if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for subset in combinations(topn_models, r):
            subset = list(subset)
            specs.append({"family": "aggregate", "name": f"subset{r}_mean", "models": subset, "agg_method": "mean"})
            specs.append({"family": "aggregate", "name": f"subset{r}_median", "models": subset, "agg_method": "median"})
            if r >= 5:
                specs.append({"family": "aggregate", "name": f"subset{r}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            specs.append({"family": "weighted", "name": f"subset{r}_softmax", "models": subset, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
            specs.append({"family": "weighted", "name": f"subset{r}_nnls", "models": subset, "weight_scheme": "nnls"})
            specs.append({"family": "weighted", "name": f"subset{r}_simplex", "models": subset, "weight_scheme": "simplex_mae"})

# Жадный прямой отбор.
if RUN_GREEDY_SEARCH:
    specs.extend(greedy_forward_specs(ranked_models))

# Stacking по top-k и по полному пулу.
if RUN_STACKERS:
    stackers = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr"]
    if HAS_XGB:
        stackers.append("xgb")
    for stacker_name in stackers:
        specs.append({"family": "stacker", "name": f"all_{stacker_name}", "models": all_models, "stacker_name": stacker_name})
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            specs.append({"family": "stacker", "name": f"top{k}_{stacker_name}", "models": topk, "stacker_name": stacker_name})

print(f"Всего конфигураций ансамблей: {len(specs)}")

# 3. Оценка ансамблей на selection-части validation
search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 200 == 0:
        print(f"Selection scoring {idx}/{len(specs)} ...")
    try:
        row = evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
        search_rows.append(row)
    except Exception as e:
        search_rows.append({
            "tag": spec_tag(spec),
            "family": spec["family"],
            "name": spec["name"],
            "n_models": len(spec["models"]),
            "models": spec["models"],
            "selection_MAE": np.inf,
            "selection_RMSE": np.inf,
            "selection_MdAE": np.inf,
            "error": repr(e),
            "spec": spec,
        })

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "error": r.get("error", ""),
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "selection_MdAE"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
search_df.to_csv(RUN_DIR / "ensemble_search_leaderboard.csv", index=False)
search_df.to_csv(ART_DIR / "ensemble_search_leaderboard_latest.csv", index=False)

top_refit_rows = [r for r in search_rows if np.isfinite(r["selection_MAE"])][:min(REFIT_TOP_ENSEMBLES, len(search_rows))]
print(f"Отобрано строк для финальной holdout-оценки: {len(top_refit_rows)}")


In [ ]:
# 4. Финальная оценка лучших ансамблей на holdout
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Финальная оценка ансамбля {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
final_df.to_csv(RUN_DIR / "ensemble_holdout_leaderboard.csv", index=False)
final_df.to_csv(ART_DIR / "ensemble_holdout_leaderboard_latest.csv", index=False)

best_ensemble_row = final_rows[0]
best_single_model = min(available_models, key=lambda m: mae(y_val, val_pred_df[m].values))

single_typ = calc_reg_metrics(y_test_typ, test_typ_fullfit_pred_df[best_single_model].values)
single_full = calc_reg_metrics(y_test_full, test_full_fullfit_pred_df[best_single_model].values)

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_fit_frac": BLEND_FIT_FRAC,
    "available_models": available_models,
    "best_single_model_by_val": best_single_model,
    "best_single_model_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_model_by_val", "value": best_single_model},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

with open(RUN_DIR / "best_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "best_ensemble_summary_latest.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"model": best_models, "weight": best_weights})
    wdf.to_csv(RUN_DIR / "best_ensemble_weights.csv", index=False)
    wdf.to_csv(ART_DIR / "best_ensemble_weights_latest.csv", index=False)

run_manifest = {
    "run_name": RUN_NAME,
    "data_path": str(DATA_PATH),
    "tuned_summary_path": str(TUNED_SUMMARY_PATH),
    "artifacts_root": str(ART_DIR),
    "run_dir": str(RUN_DIR),
    "duration_cap": DURATION_CAP,
    "train_full_rows": int(len(train_full)),
    "train_typical_rows": int(len(train_typical)),
    "inner_blend_train_rows": int(len(X_blend_tr)),
    "inner_blend_val_rows": int(len(X_blend_val)),
    "test_typical_rows": int(len(test_typical)),
    "test_full_rows": int(len(test_full)),
    "selected_models": available_models,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "search_specs_total": int(len(specs)),
    "search_specs_refit": int(len(top_refit_rows)),
    "best_single_model_by_val": best_single_model,
    "best_ensemble_tag": best_ensemble_row["tag"],
}
with open(RUN_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "run_manifest_latest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)

print("Эксперимент с ML-ансамблями завершён. Основные артефакты:")
print(RUN_DIR / "ensemble_search_leaderboard.csv")
print(RUN_DIR / "ensemble_holdout_leaderboard.csv")
print(RUN_DIR / "best_ensemble_summary.json")


In [ ]:
# 5. Графики и итоговые таблицы
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1])
plt.xlabel("MAE_typical")
plt.ylabel("Ансамбль")
plt.title("Top-15 ML-ансамблей: selection_MAE и typical holdout")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(7, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Корреляция validation-предсказаний tuned ML-моделей")
plt.tight_layout()
plt.show()

display(final_df.head(20))

near_ties = final_df[final_df["selection_MAE"] <= final_df["selection_MAE"].min() + 0.5].copy()
display(near_ties.head(20))
